<a href="https://colab.research.google.com/github/davidsmall-Durham/EDS-Practicals/blob/main/Practical_6_Part_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environmental Data Science Practical 6, Part 1: What are the limits of a sequential workflow?


In the previous practial we began working with multi-dimensional data using Python in Google Colab. We learned about  NetCDF file structure, extracting sub-sets of larger data sets, and performing simple geospatial calculations. This practical will continue with these general themes but on a larger scale.

Specifically, in Part I we will be working with daily precipitation data generated by the Australian ACCESS-CM2 climate model under the high-emission SSP5-8.5 scenario, tracking a 24-year window from 2027 to 2050. Using this daily data we will calculate a simple climate index and compare the year 2050 to a historical baseline.  

The first part of this practical aims to get you thinking in more depth about two key concepts.

1. **The dimensionality of datasets and how they are reduced**. Learning to manipulate data structured across `time`, `latitude`, and `longitude` coordinates using `xarray`
2. **The scale of datasets and the complexity of operations**. Evaluating when a dataset or a workflow becomes too large or computationally intensive for a standard local setup.

In Part II of this practical we will explore an alternative approach to undertaking geospatial analysis of big data sets while still retaining the Python/Google Colab interface which is hopefully becoming more familiar.

In addition to common Python packages we will use many of same tools as before, specifically `xarray`, the `cdsapi`, and `Cartopy` for plotting maps

*A Note on Resource Awareness*: Keep an eye on the RAM and Disk counters in the top-right corner of your Google Colab interface. As we load, convert, and work with millions of data points, watch how the virtual hardware reacts.


### Working through the practical
To work through this notebook, firstly:

1. **COPY THE NOTEBOOK TO YOUR GOOGLE DRIVE** using the "Copy to Drive" button at the top of the page.
2. Read through the instructions and execute each code block cell to see what it does. If you can't see codeblocks click on the arrow next to the Heading.
3. Answer the questions in each section and tackle the tasks that require you to write (very short) bits of code.

## **Initial Setup**

Before getting started we need to install the required packages to our environmment and mount our Google Drive.

In [ ]:
# @title ##### Install anything that isn't native to Colab
!pip install xarray cartopy netcdf4 cdsapi h5netcdf -q


In [2]:
# @title ##### Import libraries into the Colab environment
# Copernicus Data Service API
import cdsapi
import os

# Import libraries
import zipfile
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Libraries for plotting and visualising data
import cartopy.crs as ccrs
import cartopy.feature as cfeature


## **Section 1: Mounting our Google Drive**

As we did in the previous practical in this notebook we will work from our Google Drive. When we are working with relatively small files this is pretty seamless but as we scale up workflows it can produce bottlenecks due to network read/write speeds and memory limits.

In [ ]:
#@title ##### Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title ##### Define and make working directory for this practical
# define and create and dedicated folder for this practical


# Create the folder automatically if it doesn't exist yet




## **Section 2: Downloading the data**

Before we do anything we need to have some data to work with. As we did in the previous practical we are going to download some data from the CDS using the Python API. You will need to set up the `URL` and `API` key in exactly the same way as the previous practical by visiting [**this page**](https://cds.climate.copernicus.eu/how-to-api), copying your key and pasting it below.

In [ ]:
#@title ##### **Input required** This codecell sets the URL and add your key for the CDS


URL = 'https://cds.climate.copernicus.eu/api'

KEY =  # Input your personal key here

We are going to work with **daily** **global** precipitation data. However unlike before this is not observational data. We are downloading output from a General Circulation Model (GCM) that is used by scientists to make predictions of future climate under various global warming scenarios. This output formed part of a major project, the Sixth Coupled Model Intercomparison Project (CMIP6) that coordinated dozens of climate model groups producing predictions to underpin the most recent IPCC report in 2021. You can read more about CMIP6 and its role in the IPCC [here](https://www.carbonbrief.org/cmip6-the-next-generation-of-climate-models-explained/). Specifically, we want to download the output variable (**precipitation**), of one climate scenario (**SSP5_8.5**),from one model (**ACCESS-CM2**) for **all months** in the years **2027-2050**.

**Go to [this page](https://cds.climate.copernicus.eu/datasets/projections-cmip6?tab=download) and generate the API request code for the highlighted parameters.**



Depending on the traffic on the Copernicus servers, it will take several minutes to access the archive, compresses it, and make it ready for download.

You may have noticed in the previous practical that the code performs a check to see if the file set already exists in the directory before downloading. This is good coding practise as, without it, we would download the file every time we re-ran the notebook.


In [ ]:
#@title ##### This codecell defines the file path and downloads the data

# Define an informative filename for the file to be downloaded

# Now define the full path name (the address for the file)


# First we will check if the file already exists (e.g., if you run this notebook
# again there is no point waiting to donwload the file again!)


    # Now we enter the code generted by the API request on the CDS


    # Pass the URL and KEY variables to the Client constructor


Because the CDS delivers this data as a compressed .zip bundle, we need to unzip it inside our defined `DATADIR` folder before `xarray` can work with it. We use the `zipfile` package that we imported in our setup. We will do another check to see if this task has already been done. We can also check how large the file is (useful for thinking about data management)

In [ ]:
# @title ##### This codecell unzips the file

# Extract everything if not extracted yet


#calculate uncompressed size


### **Questions**

1.  How long did it take you to download this data set?
2.  If a robust IPCC-style assessment requires analysis of 30 models, 4 emissions scenarios, and a 150-year time window calculate the approximate total data volume (in GB) based on this file size.



## **Section 3: Loading and inspecting the dataset**

We have now downloaded and unzipped our modelled precipitation data into our Google Drive and are ready to start working on it. As before we will load and inspect the dataset using the `.open_dataset` command from `xarray`.

We need to tell `xarray` the path to follow to find our data set. If we don't do this properly we will get an error (*hint hint*).



In [ ]:
#@title ##### This codecell loads the data as a dataset (ds) using xarray


# Print the dataset metadata structure to inspect
print("\n=== DATASET METADATA ===")
print(ds)

As before this a useful start in understanding the structure of our dataset. We can see the dimensions of our dataset as well as the data variables. Geometrically our data is 3-D datacube defined by the dimensions `time`, `latitude` and `longitude`.

You will notice a extra 'bnds' dimensions. These are auxiliary dimensions and are used to store the start and end limits of a spatial grid cell or a temporal tracking window (i.e. the bounds). You don't  need to worry about the 'bnds' dimension directly; `xarray` keeps it in the background as metadata so it knows exactly how to align maps and graphs when you plot them. **For our purposes the three dimensions are time, latitude and longitude**. These are what we see when we print the shape of our `pr` variable.

In the code cell below write some code to print out information that helps you answer questions 1. and 2. below.

Questions 3. and 4. require you to think more conceptually about the shape of our data cube and how we can **slice** it.

In [ ]:
#@title ##### Inspect the attributes of the precipiation (pr) variable


### **Questions**

1.  What are the units for precipitation?
2.  What are the dimensions? How many *individual* precipitation data points are in this dataset?
3.  If we take a single slice along the `time` dimension what is the geometry of our selection?
4.  If we take a single slice along both spatial axes simultaneously what is the geometry of our selection?.

## **Section 4: Working with the data**


**Climate indices** have been developed to give information on variables such as precipitation. See [here](https://www.cen.uni-hamburg.de/en/icdc/data/climate-indices.html). Indices condense large, daily time-series datasets into physically meaningful metrics. By calculating indices like Maximum 1-Day Precipitation ($Rx1day$) or Heavy Precipitation Days ($R20mm$), we transform raw data into actionable metrics that matter to (for example):

*   **Civil Engineers**: Designing drainage networks, dams, and flood defenses.
*   **Farmers**: Predicting crop stress, soil erosion, and changing planting windows.


Calculating different indices involves different maths which directly impacts the computational efficiency of our code. For example, evaluating a fixed-threshold index, such as **counting** how many days have rainfall >20 mm, is a simple operation that requires minimal memory. Conversely, calculating a relative index like days exceeding historical 95th percentile introduces significant computational overhead because it requires taking a baseline period (e.g., 1981–2010), loading the time series for every grid cell, and **sorting** the daily values in ascending order to establish a threshold. Then we can "broadcast" this threshold across future projections to count exceedances. This sorting process is memory-intensive and scales poorly with large datasets. As a result, **when designing any data workflow you must consider 1) the complexity of the operations and 2) the scale of the data set(s) you are working with.**

We will keep things very simple and use the $R20mm$ Index. $R20mm$ is calculated by counting the number of days where rainfall exceeds $20 \space mm$. This is a very simple metric but it has been widely used.

---
### Converting the units
Before we do anything we need to convert the units from $kg \cdot m^{-2} \cdot s^{-1}$ to $mm \cdot day^{-1}$.  To do this we multiply every data point by $86,400$.

This is because $1 \space kg$ of water spread over $1 \space m^{2}$ is $1 \space mm$ deep (ignoring  effects of temperature on the water density), therefore $1 \space kg \cdot m^{-2} \cdot s^{-1}$ is equivalent to $1 \space mm \cdot s^{-1}$. We multiply by $86,400$ because there are $86,400$ seconds in a day.

We wil do this by creating a new variable, `pr_mm_day`, which inherits the dimensions and metadata of the original `pr` variable.

When we multiply `xarray` broadcasts the calculation across the whole 3D data array (i.e., it performs the same operation on every data point). This is quite a large number of operations and will take some time....  

In [9]:
# @title ##### **Input needed** This code cell converts units
# We will create a new variable which inherits the dimensions and metadata of the original one

# Can you write some code to convert the precipitation variable (pr) in our dataset (ds)? We will create a new variable called pr_mm_day
pr_mm_day = # <------------ Write your code here

# We will update the metadata for the pr_mm_day variable we have just created
pr_mm_day.attrs['units'] = 'mm/day'
pr_mm_day.attrs['long_name'] = 'Precipitation (mm/day)'

 After we finish converting the units to mm/day , it’s good practice to make sure the numbers make sense. We can find the maximum rainfall value in the entire dataset to check it makes sense.

In [ ]:
# @title ##### **Input needed** This codecell finds the maximum value in our new pr_mm_day dataset
#
# Hint: use the .max() function on the new variable.
max_rainfall = # <------------ WRITE YOUR CODE HERE

print(f"The absolute heaviest day of rainfall in the model is: {max_rainfall.values:.1f} mm/day")

#### **Questions**

1.  How long did it take to convert the units?
2.  How many individual operations did we just carry out?
3.  What is the maximum daily rainfall amount? Is this reasonable?


---
### Calculating the $R20 mm$ index
Now that we have converted to the correct units we can calcualte the $R20mm$ index for our data.

If we break this down into simple terms what we are asking Python to do is look at every grid cell and count the number of days per year where rainfall exceeds our $20 \space mm$ threshold.

If we were to do this by hand we would have to:

1.  Go to the very first grid cell.
2.  Check if the Day 1 value is $\ge 20 \space mm$. If it is, make a tally mark.
3.  Move to Day 2, check the value, and update the tally.
4.  Repeat this for all 365 days of the year...*just for that grid cell*.
5. Move to the next grid cell and repeat the entire process....*for every grid cell AND then every year*.

Clearly that would be impossible or at least unbelieveably slow/boring.

Even for a computer, following this step-by-step sequential process (using loops) is highly inefficient. Instead, we use a technique called vectorisation, which tells the processing unit to perform the exact same maths across the entire datacube all at once.

In this case we ask Python to look at the whole datacube and replace values above our threshold with $1$ and values below it with $0$. This is creates a Logical (or Boolean) Mask of $1's$ and $0's$ with exactly the same dimensions as our `pr_mm_day` variable. We can then `.groupby('time.year')` and `sum` up our $1's$ for each grid cell (summing $1's$ is the same as counting occasions our condition is met). The ouput is a new 3-D array `R20mm` with the same spatial dimensions but we have collapsed the time dimension into 24 annual 'layers' representing a count of the $R20mm$ index in every grid cell for each year.


In [11]:
#@title ##### Calculate the R20mm for our dataset

# First we turn our precipitation data into a boolean mask using a logical condition (i.e. the answer is either TRUE of FALSE)
# where True = 1 and False = 0, this makes it easy to count the number of days


# Now we can count for each year by grouping the daily data by year and summing


# Update Metadata
# It is good practice to update the metadata after modifications.



Again, we can check the output of our operation to see if it makes sense. In this case we want to see fluctuating, normal looking 'counts' of integers (whole numbers) and not, for example, ever-increasing floating point numbers (which would suggest we have made an error somewhere).

In [ ]:
#@title ##### This codecell checks our output.


If that looks right we can further sense-check and visualise our $R20mm$ index by selecting a single year and plotting it. We will plot the year $2050$.

In the code cell below write a line of code to select (`.sel()`) the year $2050$

In [ ]:
#@title ##### Plotting the 2050 $R20mm$ index

# Select (.sel) the year 2050 from your R20mm array
R20mm_2050 = #<-----------Write your code here

# Set up the plot size
fig = plt.figure(figsize=(12, 6))

# Define the map projection
# ccrs.PlateCarree() is standard
ax = plt.axes(projection=ccrs.PlateCarree())

# And now we plot...
# 'Transform' tells Cartopy what coordinate system the input data is in (usually regular lat/lon, so PlateCarree)
im = R20mm_2050.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap='YlGnBu',  # Yellow-Green-Blue is great for rainfall intensity
    robust = True,
    cbar_kwargs={
        'label': 'No. of days with precipiation >20 mm/day',
        'orientation': 'horizontal',
        'pad': 0.08,
        'shrink': 0.7
    }
)

# Add geographic context so we can see where the rain is
ax.coastlines(resolution='110m', color='black', linewidth=1)
ax.add_feature(cfeature.BORDERS, linestyle=':', edgecolor='gray')
ax.add_feature(cfeature.LAND, facecolor='none', edgecolor='none')

# Add gridlines and axis labels
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False   # Clean up the top axis
gl.right_labels = False # Clean up the right axis

# Add title-
plt.title('Number of days exceeding 20 mm/day (R20mm) - Year 2050', fontsize=14, pad=15)

plt.show()

From this plot we can see that the area around the equator exeperiences the highest number of days with heavy rainfall. This is what we would expect, again giving us confidence that code is working as it should.

#### **Questions**

1.  In addition to our original `pr` variable how many other (intermediate) variables (with the same dimensions) have we created up to this point? What consequences does this have for memory?

---
### **TASK!!** Comparison to a Historical Baseline (1981-2010)

It might be more informative to know how many extra heavy rainfall days are expected by 2050. As in the last practical we can compare values to a baseline to see how they have changed (or in this case are predicted to change).

CMIP models such as ACCESS2_CM2 also produce simulations for the historical period (i.e., 1850-2014) so we could compare the predicted precipitation values to those simulated for a historical baseline. We will use 1981-2010 again.

To do so we we need to download and unzip that data. To save time waiting for the CDS this zip file is provided on Learn Ultra.

Upload the file to your working folder for this practical on Google Drive and then run the code cell below. For completeness it includes all the code that is needed to download and unzip the file. However, because we include code to check if the file already exists it will skip the download/unzipping.

In [ ]:
#@title ##### Download historical data (This code does the download but you don't need to run it [takes several minutes])
# Define an informative filename for the file to be downloaded
filename_2 = "access_cm2_precip_historical.zip"

# Now define the full path name (the address for the file)
full_path_2 = os.path.join(DATADIR, filename_2)

# Again we will check if the file already exists
if os.path.exists(full_path_2):
    print("File already exists! Skipping the CDS download entirely.")
else:
    print("File not found. Connecting to CDS to download...")

# API request code
    dataset_2 = "projections-cmip6"
    request_2 = {
    "temporal_resolution": "daily",
    "experiment": "historical",
    "variable": "precipitation",
    "model": "access_cm2",
    "year": [
        "1981", "1982", "1983","1984","1985", "1986", "1987","1988", "1989", "1990","1991", "1992",
        "1993","1994", "1995", "1996","1997", "1998", "1999","2000", "2001", "2002","2003", "2004",
        "2005","2006", "2007", "2008","2009", "2010"
    ],
    "month": [
        "01", "02", "03","04", "05", "06","07", "08", "09","10", "11", "12"
    ]
}
    # Now pass to the client constructor
    client = cdsapi.Client(url=URL, key=KEY)
    client.retrieve(dataset_2, request_2).download(full_path_2)


# UNZIP CHECK
# Extract everything if not extracted yet
with zipfile.ZipFile(full_path_2, "r") as zip_ref:
    # Get the name of the first .nc file inside
    target_nc_filename_2 = [f for f in zip_ref.namelist() if f.endswith(".nc")][0]
    nc_file_path_2 = os.path.join(DATADIR, target_nc_filename_2)

    if not os.path.exists(nc_file_path_2):
        print(f"Extracting {target_nc_filename_2}...")
        zip_ref.extractall(DATADIR)
    else:
        print(f"File '{target_nc_filename_2}' already extracted.")

size_gb = os.path.getsize(nc_file_path_2) / (1024**3)
print(f" -> Uncompressed Size: {size_gb:.2f} GB")


Now we want to slice our data sets to a specific area of interest, you could choose any part of the world you like but one suggestion would be to look at the area affected by the Indian Monsoon;
* **Latitude:** $0^\circ \text{N}$ to $45^\circ \text{N}$
* **Longitude:** $60^\circ \text{E}$ to $140^\circ \text{E}$

Remember how we used `.sel` in thge first notebook last week?

In [ ]:
#@title #### Use this code cell to open the new historical data set and slice it to you AOI

# Open the dataset using xarray


# Now slice the historical data to your chosen AOI


# We also need to slice our future predictions dataset (we can just use the R20mm grid for 2050)


Just like before we will also need to convert the units for the historical data.

Remember we need to open the dataset with `xarray` first!

In [ ]:
# @title ##### Use this cell to convert the units



# We also need to update the metadata for the pr_mm_day_hist variable


To generate our historical baseline we need to calculate $R20mm$ across every grid cell for every year in the historical period and then take a mean for each individual grid cell.

**Question**
1. Before you calculate the historical $R20mm$, what geometry would you expect the output to have?



In [ ]:
#@title ##### **Input Needed** Use this codecell to compute the historical R20mm.

# First turn the precipitation data into a boolean mask
# where True = 1 and False = 0, this makes it easy to count the number of days


# Now count for each year by grouping the daily data by year and summing along the time dimension


# Update Metadata


# Can you write the code to collapse the 3D data cube into a
# mean R20mm for the whole baseline period?
# HINT: We are averaging over the 'year' dimension


# Update metadata to reflect that this is now an averaged climatology


# Print out some details of the variable to sense check



The historical $R20mm$ is a 2-D array beacuse we have averaged across our time dimension but have not reduced our spatial dimensions.

We now have our baseline and our values for 2050 so we can calculate our anomaly.

Can you calculate the anomaly (2050 - historical baseline)?

In [ ]:
#@title ##### **Input Needed** Calculate the anomaly and then plot it

anomaly_2050= # <----------- Write your code here

# Set up the plot size

# Define the map projection
# ccrs.PlateCarree() is standard


# And now we plot... 'transform' tells Cartopy what coordinate system the input
# data is in (usually regular lat/lon, so PlateCarree is good)


# Add geographic context so we can see where the rain is

# Add gridlines and axis labels


# Add title


# And show

Looking at the map above, do you think the colour scheme helps conceptualise the data?

In the code cell above change the colour scheme to one that is more intuitive to visualise the anomaly

For a list of all available schemes see [this link](https://matplotlib.org/stable/users/explain/colors/colormaps.html).

## **Summary and reflections**



We successfully processed one variable (`precipitation`) from one model (`ACCESS-CM2`) for one scenario (`SSP5-8.5`) over a 24-year window, resulting in a ~0.82 GB dataset. To calculate an anomaly we also had to process the corresponding historical baseline. Relying on an individual model introduces significant structural uncertainty because every model simplifies complex physical processes (e.g., cloud formation, convective rainfall, and albedo) slightly differently.

To conduct a robust climate impact assessment, practitioners utilise **multi-model ensembles**, averaging multiple models across multiple **emissions** pathways (e.g., SSP2-4.5 vs. SSP5-8.5) up to the year 2100.

**Look back at your answers throughout this practical, and reflect on the following:**

1. **Computational Bottlenecks:** Is a standard sequential workflow (downloading `.zip` files, unzipping to disk, and loading them into memory via a local/single-instance runtime) viable for an ensemble analysis that spans tens of terabytes?
2. **Alternative platforms** If working locally or using a standard single-instance virtual machine is unmanageable, what modern environmental data science platforms exist to solve this?
3. **Multidimensional Thinking:** When computing a multi-model ensemble mean, you aren't just aggregating data over time or space; you are adding an entirely new **dimension** (the 'Model' dimension) to your data cube. What additional complexities might this introduce?